# Bronze Layer Ingestion - Provider Drug Data
## Healthcare Lakehouse Project

This notebook ingests the **Provider Drug dataset** (~3.5 GB) from S3 into the Bronze layer using **Databricks Auto Loader**.

### Key Features:
- **Auto Loader (cloudFiles)**: Efficiently processes large volumes of new files
- **Schema Evolution**: Handles new columns with `addNewColumns` mode
- **Data Quality Checks**: Row count validation, null checks, duplicate detection
- **Metadata Tracking**: Captures file name and size for lineage
- **Delta Lake**: Writes to Delta format for ACID transactions

### Dataset Size: ~3.5 GB

### Partitioning Strategy:
- **Bronze**: `read_timestamp` for ingestion batch tracking
- **Silver**: `prscrbr_state_abrvtn` for query optimization (applied in silver layer)

### Step 1: Import Required Libraries

In [ ]:
# Databricks notebook source
from pyspark.sql import functions as F
from delta.tables import DeltaTable

### Step 2: Load Utility Variables
Runs the utilities notebook to access schema name variables.

In [ ]:
MAGIC
%run /Workspace/Users/pawanvirat32@gmail.com/healthcare_lakehouse_project/01_setup/utilities

### Step 3: Verify Schema Variables
Confirms that the schema variables are loaded correctly.

In [ ]:
print(bronze_schema, silver_schema, gold_schema)

### Step 4: Define Widget Parameters
Creates interactive widgets for runtime configuration:
- **catalog**: Unity Catalog name (default: healthcare_lakehouse)
- **data_source**: Source system identifier (default: cms)

In [ ]:
dbutils.widgets.text('catalog', 'healthcare_lakehouse', 'Catalog')
dbutils.widgets.text('data_source', 'cms', 'Data Source')

### Step 5: Get Widget Values
Retrieves the widget values and displays them for verification.

In [ ]:
catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')

print(f"catalog: {catalog}")
print(f"data_source: {data_source}")

### Step 6: Configure Auto Loader Stream for Drug Data
Sets up the Auto Loader configuration for the larger drug dataset:
- **cloudFiles.schemaEvolutionMode**: "addNewColumns" for automatic schema evolution
- **maxBytesPerTrigger**: 128MB chunks (smaller due to larger dataset size)
- **Schema Location**: Tracks schema changes over time

In [ ]:
S3_BASE        = f"s3://healthcare-claims-lakehouse/raw/{data_source}"
CHECKPOINT_BASE = f"s3://healthcare-claims-lakehouse/checkpoints/{data_source}"
TARGET_SCHEMA  = f"{catalog}.bronze"

In [ ]:
df_drug = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", f"{CHECKPOINT_BASE}/by_provider_drug/schema")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("cloudFiles.maxBytesPerTrigger", "128m")
    .load(f"{S3_BASE}/by_provider_drug/")
    .withColumn('read_timestamp', F.current_timestamp())
    .select('*', '_metadata.file_name', '_metadata.file_size')
)

### Step 7: Data Quality Checks
Validates data before writing to ensure quality and integrity:
- **Row Count**: Validates expected volume
- **Null Checks**: Critical columns (Prscrbr_NPI, Gnrc_Name, Tot_Clms)
- **Duplicate Detection**: Identifies duplicate records by key columns

In [ ]:
# Data Quality Check Function
def validate_drug_data(df):
    """Validate drug ingestion data quality"""
    
    # Row count
    total_rows = df.count()
    print(f"Total rows ingested: {total_rows:,}")
    
    # Null checks on critical columns
    null_checks = df.select([
        F.count(F.when(F.col("Prscrbr_NPI").isNull(), 1)).alias("null_prscrbr_npi"),
        F.count(F.when(F.col("Gnrc_Name").isNull(), 1)).alias("null_gnrc_name"),
        F.count(F.when(F.col("Tot_Clms").isNull(), 1)).alias("null_tot_clms"),
        F.count(F.when(F.col("Tot_Drug_Cst").isNull(), 1)).alias("null_tot_drug_cst")
    ])
    print("Null checks on critical columns:")
    null_checks.show(truncate=False)
    
    # Duplicate detection
    duplicates = df.groupBy("Prscrbr_NPI", "Gnrc_Name", "Brnd_Name").count().filter("count > 1")
    dup_count = duplicates.count()
    print(f"Duplicate records found: {dup_count:,}")
    
    # File metadata
    print("\nFile metadata:")
    df.select("file_name", "file_size", "read_timestamp").distinct().show(truncate=False)
    
    return total_rows, dup_count

# Run validation (batch mode for preview)
# Note: For streaming, validation happens per micro-batch

### Step 8: Write to Bronze Delta Table with Partitioning
Writes the streaming data to the Bronze layer Delta table:
- **Partition By**: `read_timestamp` for ingestion batch tracking
  - Silver layer repartitions by `prscrbr_state_abrvtn` for query optimization
- **Checkpoint Location**: Tracks processing progress
- **mergeSchema**: Allows schema evolution
- **trigger.availableNow**: Processes all available data and stops

In [ ]:
# Write to Bronze Delta table
(
    df_drug.writeStream
    .format("delta")
    .option("checkpointLocation", f"{CHECKPOINT_BASE}/by_provider_drug/checkpoint")
    .option("mergeSchema", "true")
    .partitionBy("read_timestamp")  # Bronze: ingestion batch tracking
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(f"{TARGET_SCHEMA}.by_provider_drug")
)

### Step 9: Preview Data and Validation Results (Batch Mode)
Reads the data in batch mode to preview the ingested records and run validation.

In [ ]:
# Preview data by reading in batch mode
df_preview = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(f"{S3_BASE}/by_provider_drug/")
)

In [ ]:
display(df_preview.limit(30))

In [ ]:
# Run data quality validation
total_rows, dup_count = validate_drug_data(df_preview)
print(f"\nValidation Complete: {total_rows:,} rows, {dup_count:,} duplicates")